<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

##### 模块 2.6：更多关于 ChiselTest
**上一步：[融会贯通：一个 FIR 滤波器](2.5_exercise.ipynb)**<br>
**下一步：[生成器：参数](3.1_parameters.ipynb)**

## 动机
Chisel 团队一直在致力于改进测试框架。“ChiselTest”提供了以下改进。

- 适用于单元测试和系统集成测试
- 专为可组合抽象和分层而设计
- 高度可用，通过使其尽可能简单、轻松（避免样板代码和其他无意义的内容）和有用，鼓励编写单元测试

### 计划中
- 能够针对多个后端和仿真器（如果测试向量不是静态的，则可能需要链接到 Scala，或者在综合到 FPGA 时使用有限的测试构建 API 子集）
- 将包含在基础 chisel3 中，以避免打包和依赖项噩梦


## 设置

In [ ]:
val path = System.getProperty("user.dir") + "/source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.experimental._
import chisel3.experimental.BundleLiterals._
import chisel3.tester._
import chisel3.tester.RawTester.test

>本训练营需要一些与您在其他地方看到的 chisel 导入略有不同的导入。
`import chisel3.tester.RawTester.test` 引入了
下面专门为训练营设计的 `test(...)` 版本。

---
# 基本测试器实现

ChiselTest 从与 iotesters 相同的基本操作开始。以下是旧 iotesters 和新 ChiselTest 之间基本
功能映射的简要总结

|        | iotesters             | ChiselTest            |
| :----  | :---                  | :---                |
| poke   | poke(c.io.in1, 6)     | c.io.in1.poke(6.U)    |
| peek   | peek(c.io.out1)       | c.io.out1.peek()      |
| expect | expect(c.io.out1, 6)  | c.io.out1.expect(6.U) |
| step   | step(1)               | c.io.clock.step(1)  |
| initiate | Driver.execute(...) { c => | test(...) { c => |


让我们从 2.1 中的简单直通模块开始。

In [ ]:
// Chisel 代码，但传入一个参数来设置端口宽度
class PassthroughGenerator(width: Int) extends Module { 
  val io = IO(new Bundle {
    val in = Input(UInt(width.W))
    val out = Output(UInt(width.W))
  })
  io.out := io.in
}

使用旧样式，一个简单的测试如下所示：

```scala
val testResult = Driver(() => new Passthrough()) {
  c => new PeekPokeTester(c) {
    poke(c.io.in, 0)     // 将我们的输入设置为值 0
    expect(c.io.out, 0)  // 断言输出正确地为 0
    poke(c.io.in, 1)     // 将我们的输入设置为值 1
    expect(c.io.out, 1)  // 断言输出正确地为 1
    poke(c.io.in, 2)     // 将我们的输入设置为值 2
    expect(c.io.out, 2)  // 断言输出正确地为 2
  }
}
assert(testResult)   // Scala 代码：如果 testResult == false，将抛出错误
println("成功！！") // Scala 代码：如果我们到达这里，我们的测试通过了！
```



In [ ]:
test(new PassthroughGenerator(16)) { c =>
    c.io.in.poke(0.U)     // 将我们的输入设置为值 0
    c.io.out.expect(0.U)  // 断言输出正确地为 0
    c.io.in.poke(1.U)     // 将我们的输入设置为值 1
    c.io.out.expect(1.U)  // 断言输出正确地为 1
    c.io.in.poke(2.U)     // 将我们的输入设置为值 2
    c.io.out.expect(2.U)  // 断言输出正确地为 2
}

>为了说明 ChiselTest 推进时钟的方式，我们可以
在前面的示例中添加一些 `step` 操作。

In [ ]:
test(new PassthroughGenerator(16)) { c =>
    c.io.in.poke(0.U)     // 将我们的输入设置为值 0
    c.clock.step(1)    // 推进时钟
    c.io.out.expect(0.U)  // 断言输出正确地为 0
    c.io.in.poke(1.U)     // 将我们的输入设置为值 1
    c.clock.step(1)    // 推进时钟
    c.io.out.expect(1.U)  // 断言输出正确地为 1
    c.io.in.poke(2.U)     // 将我们的输入设置为值 2
    c.clock.step(1)    // 推进时钟
    c.io.out.expect(2.U)  // 断言输出正确地为 2
}

---
## 上述示例中需要注意的事项

ChiselTest 的 `test` 方法需要的样板代码更少。以前的 `PeekPokeTester` 现在
已内置到流程中。

`poke` 和 `expect` 方法现在是每个 `io` 元素的组成部分。
这为测试器提供了重要的提示，以便更好地检查类型。
`peek` 和 `step` 操作现在也是 `io` 元素的方法。

另一个区别是，poke 和 expect 的值是 Chisel 字面量。
虽然这里很简单，但在更高级和有趣的示例中，它也提供了更强的检查。
随着即将到来的对指定 `Bundle` 字面量能力的改进，这一点将得到进一步增强。



# 具有解耦接口的模块
在本节中，我们将介绍一些 tester2 用于处理 `Decoupled` 接口的工具。
`Decoupled` 接受一个 chisel 数据类型，并为其提供 `ready` 和 `valid` 信号。
ChiselTest 提供了一些不错的工具来自动化和可靠地测试这些接口。

## 一个队列示例
`QueueModule` 传递数据，其类型由 `ioType` 决定。`QueueModule` 内部有 `entries` 个状态元素，这意味着它可以在施加反压之前容纳那么多元素。

In [ ]:
class QueueModule[T <: Data](ioType: T, entries: Int) extends MultiIOModule {
  val in = IO(Flipped(Decoupled(ioType)))
  val out = IO(Decoupled(ioType))
  out <> Queue(in, entries)
}

## EnqueueNow 和 expectDequeueNow
*ChiselTest* 内置了一些处理 IO 中解耦接口电路的方法。在本例中，我们将了解如何向 `queue` 中插入和提取值。

| 方法 | 描述 |
| :---   | :---        |
| enqueueNow | 向 `Decoupled` 输入接口添加（入队）一个元素 |
| expectDequeueNow | 从 `Decoupled` 输出接口移除（出队）一个元素 |
---


>注意：需要一些必需的样板代码 `initSource`、`setSourceClock` 等，以确保 `ready` 和 `valid` 字段在
测试开始时都已正确初始化。


In [ ]:
test(new QueueModule(UInt(9.W), entries = 200)) { c =>
    // 显示队列使用和行为的示例测试序列
    c.in.initSource()
    c.in.setSourceClock(c.clock)
    c.out.initSink()
    c.out.setSinkClock(c.clock)
    
    val testVector = Seq.tabulate(200){ i => i.U }

    testVector.zip(testVector).foreach { case (in, out) =>
      c.in.enqueueNow(in)
      c.out.expectDequeueNow(out)
    }
}

## EnqueueSeq 和 DequeueSeq 
现在我们将介绍两个新方法，用于处理单个操作中的入队和出队操作。

| 方法 | 描述 |
| :---   | :---        |
| enqueueSeq | 继续将 `Seq` 中的元素逐个添加到 `Decoupled` 输入接口，直到序列耗尽 |
| expectDequeueSeq | 从 `Decoupled` 输出接口逐个移除元素，并将每个元素与 `Seq` 中的下一个元素进行比较 |
---
> 注意：下面的示例可以正常工作，但是，按照编写的方式，`enqueueSeq` 必须在 `expectDequeueSeq` 开始之前完成。如果 `testVector` 的大小大于队列深度，则此示例将失败，因为队列将已满并且无法完成 `enqueueSeq`。您可以自己尝试一下，看看失败是什么样子。在下一节中，我们将展示如何正确构建此类测试。


In [ ]:
test(new QueueModule(UInt(9.W), entries = 200)) { c =>
    // 显示队列使用和行为的示例测试序列
    c.in.initSource()
    c.in.setSourceClock(c.clock)
    c.out.initSink()
    c.out.setSinkClock(c.clock)
    
    val testVector = Seq.tabulate(100){ i => i.U }

    c.in.enqueueSeq(testVector)
    c.out.expectDequeueSeq(testVector)
}

> 上一节中另一个重要的收获是，我们刚刚看到的函数 `enqueueNow`、
`enqueueSeq`、`expectDequeueNow` 和 `expectDequeueSeq` 并非 ChiselTest 中的复杂特例逻辑。
相反，它们是 ChiselTest 鼓励您从 ChiselTest 原语构建的各种测试工具的示例。要了解这些方法的实现方式，请查看 [TestAdapters.scala](https://github.com/ucb-bar/chisel-testers2/blob/d199c5908828d0be5245f55fce8a872b2afb314e/src/main/scala/chisel3/tester/TestAdapters.scala)

# ChiselTest 中的 Fork 和 Join

在本节中，我们将研究如何并发运行单元测试的各个部分。为此，我们将介绍 testers2 的两个新功能。

| 方法 | 描述 |
| :---   | :---        |
| fork   | 启动一个并发代码块，可以通过附加到前一个 fork 代码块末尾的 .fork 来并发运行其他 fork |
| join | 将多个相关的 fork 重新统一回调用线程 |
---

在下面的示例中，两个 `fork` 链接在一起，然后 `join`。在第一个 `fork` 块中，`enqueueSeq` 将继续添加元素直到耗尽。第二个 `fork` 块将在数据可用时对每个周期执行 `expectDequeueSeq`。

>由 fork 创建的线程以确定性顺序运行，主要根据它们在代码中指定的顺序，并且某些依赖于其他线程的容易出错的操作会通过运行时检查被禁止。


In [ ]:
test(new QueueModule(UInt(9.W), entries = 200)) { c =>
    // 显示队列使用和行为的示例测试序列
    c.in.initSource()
    c.in.setSourceClock(c.clock)
    c.out.initSink()
    c.out.setSinkClock(c.clock)
    
    val testVector = Seq.tabulate(300){ i => i.U }

    fork {
        c.in.enqueueSeq(testVector)
    }.fork {
        c.out.expectDequeueSeq(testVector)
    }.join()
}

## 使用 Fork 和 Join 实现 GCD
在本节中，我们将使用 fork join 方法来实现*最大公约数* **GCD** 的测试。
让我们从定义 IO 包开始。我们将在这里添加一些样板代码，以便能够使用 `Bundle` *字面量*。正如注释所说，我们希望很快就能支持自动生成字面量支持代码。

In [ ]:
class GcdInputBundle(val w: Int) extends Bundle {
  val value1 = UInt(w.W)
  val value2 = UInt(w.W)
}

In [ ]:
class GcdOutputBundle(val w: Int) extends Bundle {
  val value1 = UInt(w.W)
  val value2 = UInt(w.W)
  val gcd    = UInt(w.W)
}

现在让我们看一下 **GCD** 的 *Decoupled* 版本。我们在这里对其进行了一些修改，以使用 `Decoupled` 包装器，该包装器向输入和输出 Bundle 添加了 `ready` 和 `valid` 信号。`Flipped` 包装器接受默认创建为输出的 `Decoupled` `GcdInputBundle`，并将每个字段转换为相反的方向（递归地）。`Decoupled` 的捆绑参数的数据元素放置在顶层字段 `bits` 中。

In [ ]:
/**
  * 使用减法计算 GCD。
  * 从寄存器 x 和 y 中的较大者减去较小者，直到寄存器 y 为零。
  * 此时输入寄存器 x 的值即为 GCD
  * 返回包含两个输入值及其 GCD 的信息包
  */
class DecoupledGcd(width: Int) extends MultiIOModule {

  val input = IO(Flipped(Decoupled(new GcdInputBundle(width))))
  val output = IO(Decoupled(new GcdOutputBundle(width)))

  val xInitial    = Reg(UInt())
  val yInitial    = Reg(UInt())
  val x           = Reg(UInt())
  val y           = Reg(UInt())
  val busy        = RegInit(false.B)
  val resultValid = RegInit(false.B)

  input.ready := ! busy
  output.valid := resultValid
  output.bits := DontCare

  when(busy)  {
    // 计算过程中，不断用较大者减去较小者
    when(x > y) {
      x := x - y
    }.otherwise {
      y := y - x
    }
    when(y === 0.U) {
      // 当 y 变为零时，计算结束，
      // 如果输出准备就绪，则向输出发送有效数据
      output.bits.value1 := xInitial
      output.bits.value2 := yInitial
      output.bits.gcd := x
      output.valid := true.B
      busy := ! output.ready
    }
  }.otherwise {
    when(input.valid) {
      // 有效数据可用且没有计算正在进行，获取新值并开始
      val bundle = input.deq()
      x := bundle.value1
      y := bundle.value2
      xInitial := bundle.value1
      yInitial := bundle.value2
      busy := true.B
      resultValid := false.B
    }
  }
}

我们的测试看起来与之前的队列测试非常相似。
但是，由于计算需要多个周期，因此在计算每个 GCD 时，输入排队过程会被阻塞，所以这里有更多的事情发生。
好消息是，这方面的测试对于不同的解耦电路来说是简单且一致的。

这里还介绍了新的 Chisel3 `Bundle` 字面量表示法。考虑以下这行代码：
```scala
new GcdInputBundle(16).Lit(_.value1 -> x.U, _.value2 -> y.U)
```
上面定义的 `GcdInputBundle` 有两个字段 `value1` 和 `value2`。
我们通过首先创建一个 bundle，然后调用其 `.Lit` 方法来创建 bundle 字面量。
该方法接受一个可变参数列表，其中包含键/值对，键（例如 `_.value`）是字段名，值（例如 x.U）是 chisel 硬件字面量，Scala `Int` x 被转换为 Chisel `UInt` 字面量。
字段名前面的 `_.` 是必需的，用于将名称值绑定到 bundle 内部。

>这可能不是完美的表示法，但在广泛的开发讨论中，它被认为是
在最小化样板代码和 Scala 中可用的表示法限制之间的最佳平衡。


In [ ]:
test(new DecoupledGcd(16)) { dut =>
  dut.input.initSource().setSourceClock(dut.clock)
  dut.output.initSink().setSinkClock(dut.clock)

  val testValues = for { x <- 1 to 10; y <- 1 to 10} yield (x, y)
  val inputSeq = testValues.map { case (x, y) =>
    (new GcdInputBundle(16)).Lit(_.value1 -> x.U, _.value2 -> y.U)
  }
  val resultSeq = testValues.map { case (x, y) =>
    new GcdOutputBundle(16).Lit(_.value1 -> x.U, _.value2 -> y.U, _.gcd -> BigInt(x).gcd(BigInt(y)).U)
  }

  fork {
    dut.input.enqueueSeq(inputSeq)
  }.fork {
    for (expected <- resultSeq) {
      dut.output.expectDequeue(expected)
      dut.clock.step(5) // 在接收下一个输出之前等待一些周期以产生反压
    }
  }.join()
}


---
# 您已完成！

[返回顶部。](#top)